# Vector Stores and Retrievers

The abstractions are designed to support retrieval of data-- from (VECTOR) databases and other sources for integration with LLM workflows.
They are very important for applications that fetch data to be reasoned over as a part of model inference, as in the case of retrieval-augmented generation.


### Documents

LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- ```page_content``` : a string representing the content

- ```metadata``` : a dict containing arbitrary metadata. 

The metadata attribute can capture information about the source of the document, it's relationship to other documents and other information. Note than an individual Document object often represents a chunk of larger document.

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and playful nature.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent animals, often cherished for their grace and agility.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Parrots are colorful birds, famous for their ability to mimic human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular aquarium fish, admired for their vibrant colors and peaceful demeanor.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Rabbits are gentle creatures, often kept as pets for their soft fur and friendly behavior.",
        metadata={"source": "mammal-pets-doc"},)
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are colorful birds, famous for their ability to mimic human speech.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular aquarium fish, admired for their vibrant colors and peaceful demeanor.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are gentle creatures, often kept as pets for their soft fur and friendly behavior.')]

## Vector Stores

In [24]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

True

In [31]:
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E7B9351F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E7B93516D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [10]:
## Vector Stores

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embeddings)
vectorstore

In [14]:
vectorstore.similarity_search("cats")

[Document(id='365e4a9e-1670-4a9a-8110-4dacb971d42c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.'),
 Document(id='56a31a7a-d281-46f0-9d11-6e2e6c19fb1b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are gentle creatures, often kept as pets for their soft fur and friendly behavior.'),
 Document(id='2cba355a-b77f-4114-ae5c-c32b0a353016', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.'),
 Document(id='4e82ef80-da80-4af7-bd34-197156566190', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are colorful birds, famous for their ability to mimic human speech.')]

In [13]:
## Async Query
await vectorstore.asimilarity_search("cats")

[Document(id='365e4a9e-1670-4a9a-8110-4dacb971d42c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.'),
 Document(id='56a31a7a-d281-46f0-9d11-6e2e6c19fb1b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are gentle creatures, often kept as pets for their soft fur and friendly behavior.'),
 Document(id='2cba355a-b77f-4114-ae5c-c32b0a353016', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.'),
 Document(id='4e82ef80-da80-4af7-bd34-197156566190', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are colorful birds, famous for their ability to mimic human speech.')]

In [15]:
vectorstore.similarity_search_with_score("cats")

[(Document(id='365e4a9e-1670-4a9a-8110-4dacb971d42c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.'),
  0.9947229623794556),
 (Document(id='56a31a7a-d281-46f0-9d11-6e2e6c19fb1b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are gentle creatures, often kept as pets for their soft fur and friendly behavior.'),
  1.3951771259307861),
 (Document(id='2cba355a-b77f-4114-ae5c-c32b0a353016', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.'),
  1.4930357933044434),
 (Document(id='4e82ef80-da80-4af7-bd34-197156566190', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are colorful birds, famous for their ability to mimic human speech.'),
  1.5422139167785645)]

### Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language Chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g. Synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. 

We will build one around the ```similarity_search``` method.

In [16]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["dogs","cats"])

[[Document(id='2cba355a-b77f-4114-ae5c-c32b0a353016', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.')],
 [Document(id='365e4a9e-1670-4a9a-8110-4dacb971d42c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.')]]

VectorStores implement an ```as_retriever``` method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific ```search_type``` and ```search_kwargs``` that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following :

(Best way to query from a vector db)

In [18]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriever.batch(["dogs","cats"])

[[Document(id='2cba355a-b77f-4114-ae5c-c32b0a353016', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and playful nature.')],
 [Document(id='365e4a9e-1670-4a9a-8110-4dacb971d42c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals, often cherished for their grace and agility.')]]

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

In [38]:
# prompt = ChatPromptTemplate.from_template([("human", message)])
prompt = ChatPromptTemplate.from_messages([("human", message)])


rag_chain = {"context": retriever, "question": RunnablePassthrough() } | prompt | llm

response = rag_chain.invoke("Tell me about dogs")
print(response.content)

Dogs are great companions, known for their loyalty and playful nature.
